# NCM research run: `c4_sudoku_pilot`

Kill experiment for C4 (Recurrent NCM) on Sudoku-hard, small scale, 1 seed per arm. Trains our Sudoku tokenizer, then (a) upstream-style NCM generator and (b) the same generator with its hidden state carried across levels and trained with backprop through levels. Same params, same NFE (12). Pre-registered: (b) beats (a) by >= 3 points on our held-out puzzles, and by >= 10 points on puzzles needing >= 9 propagation rounds; otherwise C4 is killed. Released test set scored once at the end (--final), not used for selection.

- Recommended GPU: **T4 (any GPU works)**. Estimated cost: **3.0 compute units**.
- Runtime > Change runtime type > GPU, then Runtime > Run all.
- If Colab disconnects, just Run all again: every command resumes from its last Drive checkpoint.
- When done, copy `MyDrive/ncm-research/results/c4_sudoku_pilot/` into the repo's `results/colab/`.

In [ ]:
import os, subprocess, sys, json, shutil, time
T_START = time.time()
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ncm-research'
R = f'{DRIVE}/runs'; D = f'{DRIVE}/data/tinystories'
os.makedirs(R, exist_ok=True); os.makedirs(D, exist_ok=True)
REPO, BRANCH = 'https://github.com/amritisepic/NCM-research.git', 'claude/ncm-research'
CMDS = [
 "python scripts/train_sudoku.py tokenizer --config configs/sudoku/tok.yaml --level hard --data assets/sudoku_train/hard.npz --run_dir {R}/sdk_tok_hard",
 "python scripts/train_sudoku.py ncm --config configs/sudoku/gen_s.yaml --level hard --data assets/sudoku_train/hard.npz --tokenizer {R}/sdk_tok_hard/final.pt --run_dir {R}/sdk_ncm_s_hard --final",
 "python scripts/train_sudoku.py rncm --config configs/sudoku/gen_s.yaml --level hard --data assets/sudoku_train/hard.npz --tokenizer {R}/sdk_tok_hard/final.pt --run_dir {R}/sdk_rncm_s_hard --final"
]
SEEDS = [0]   # edit (e.g. [1, 2]) to run more seeds of the same commands
NEEDS = []
NEED_TS = 'ts' in NEEDS or any('scripts/train.py ' in c for c in CMDS)
if not os.path.exists('/content/NCM-research'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '/content/NCM-research'], check=True)
else:
    subprocess.run(f'cd /content/NCM-research && git fetch origin {BRANCH} && git checkout {BRANCH} && git reset --hard origin/{BRANCH}', shell=True, check=True)
os.chdir('/content/NCM-research')
print(subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'omegaconf', 'hydra-core', 'regex', 'mauve-text', 'faiss-cpu'], check=True)

In [ ]:
import urllib.request
os.makedirs('data', exist_ok=True)
if NEED_TS:
    B = 'https://huggingface.co/datasets/hazyresearch/ncm-datasets/resolve/main/tinystories/'
    for f in ['bpe_vocab_8192.pickle', 'bpe_merges_8192.pickle', 'valid_bpe_8192.pickle',
              'train_bpe_8192_offsets.npy', 'train_bpe_8192.bin']:
        p = f'{D}/{f}'
        if not os.path.exists(p):
            print('downloading', f, flush=True)
            urllib.request.urlretrieve(B + f, p + '.tmp'); os.replace(p + '.tmp', p)
    if not os.path.exists('data/tinystories'):
        os.symlink(D, 'data/tinystories')
    # local copy of the big stream for fast random access (Drive reads are slow)
    if not os.path.exists('/content/ts'):
        os.makedirs('/content/ts')
        for f in os.listdir(D):
            shutil.copy(f'{D}/{f}', f'/content/ts/{f}')
if 'ckpt' in NEEDS or 'wmt14' in NEEDS:
    ART = f'{DRIVE}/artifacts'; os.makedirs(ART, exist_ok=True)   # released checkpoints, cached on Drive
    if not os.path.exists('artifacts'):
        os.symlink(ART, 'artifacts')
    subprocess.run('bash scripts/fetch_assets.sh checkpoints', shell=True, check=True)
    if 'wmt14' in NEEDS:
        subprocess.run('bash scripts/fetch_assets.sh wmt14', shell=True, check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu'], check=True)
S = 'https://huggingface.co/datasets/hazyresearch/ncm-datasets/resolve/main/sudoku/'
for lvl in ['easy', 'medium', 'hard']:
    os.makedirs(f'data/sudoku/{lvl}', exist_ok=True)
    for f in ['puzzles.pickle', 'solutions.pickle']:
        if not os.path.exists(f'data/sudoku/{lvl}/{f}'):
            urllib.request.urlretrieve(f'{S}{lvl}/{f}', f'data/sudoku/{lvl}/{f}')
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
CMDS_RAW = list(CMDS)
for seed in SEEDS:
    for i, c in enumerate(CMDS):
        c = c.format(R=R, D=D, S=seed)
        if 'scripts/train.py ' in c or 'scripts/train_gen.py ' in c:
            c += ' --data_dir /content/ts'
        if 'configs/scale/L_ts_' in c and '--micro_batch' not in c and torch.cuda.is_available() \
                and torch.cuda.get_device_properties(0).total_memory > 39e9:
            c += ' --micro_batch 64'   # 40-80 GB GPUs: same optimizer step, fewer accumulation chunks (13.7 GB at 32)
        if '{S}' not in CMDS_RAW[i] and seed != SEEDS[0]:
            continue   # seed-independent command: run once
        print('>>>', c, flush=True)
        t = time.time()
        p = subprocess.run(c, shell=True)
        print(f'<<< exit {p.returncode} after {(time.time()-t)/60:.1f} min', flush=True)
        if p.returncode != 0:
            raise SystemExit('command failed; fix and re-run this cell (it resumes)')

In [ ]:
out = f'{DRIVE}/results/c4_sudoku_pilot'; os.makedirs(out, exist_ok=True)
for seed in SEEDS:
    for c in CMDS:
        if '--run_dir ' not in c:
            continue
        rd = c.split('--run_dir ')[1].split()[0].format(R=R, S=seed)
        name = os.path.basename(rd)
        for f in ['result.json', 'metrics.jsonl', 'config.yaml', 'calib.json', 'eval_final.json', 'eval_15k.json', 'samples_final.json']:
            if os.path.exists(f'{rd}/{f}'):
                shutil.copy(f'{rd}/{f}', f'{out}/{name}__{f}')
for f in os.listdir('results'):   # diagnostics write straight into the repo's results/
    if f.startswith(('diag_', 'b2_jacobi_', 'sudoku_', 'wmt14_')) and os.path.getmtime(f'results/{f}') > T_START:
        shutil.copy(f'results/{f}', f'{out}/{f}')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
json.dump({'run': 'c4_sudoku_pilot', 'gpu': gpu, 'finished': time.ctime()}, open(f'{out}/_meta.json', 'w'))
print('Results copied to', out, '-> please drop this folder into the repo under results/colab/')